In [1]:
# The following code will only execute
# successfully when compression is complete

import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohabenloughmari/miccai-task2-full")

print("Path to dataset files:", path)

100%|██████████| 6.72G/6.72G [05:09<00:00, 23.3MB/s]  

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/mohabenloughmari/miccai-task2-full/versions/1


In [2]:
import pandas as pd
import torch
from tqdm import tqdm
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np
from sklearn.metrics import f1_score, cohen_kappa_score, recall_score, accuracy_score, classification_report
from scipy.stats import spearmanr
from torch.utils.data import WeightedRandomSampler


import warnings
warnings.filterwarnings("ignore")

base_path = path
path = os.path.join(base_path, 'Task_2')

df_train = pd.read_csv(os.path.join(path, "df_task2_train.csv"))
df_val = pd.read_csv(os.path.join(path, "df_task2_val.csv"))
df_test = pd.read_csv(os.path.join(path, "df_task2_test.csv"))

device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')



In [3]:

labels = df_train['label'].values.astype(int)
class_counts = np.bincount(labels)
sample_weights = 1.0 / class_counts[labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

n_labels = df_train['label'].nunique()

tabular_cols = ['case', 'label', 'LOCALIZER', 'split_type', 'image']
tab_train = df_train.drop(columns=tabular_cols)
tab_val = df_val.drop(columns=tabular_cols)
tab_test = df_test.drop(columns=tabular_cols)

cat_cols = tab_train.select_dtypes(include='object').columns.tolist()
num_cols = tab_train.select_dtypes(exclude='object').columns.tolist()

print(f"Categorical columns: {cat_cols}")
print(f"Numerical columns: {num_cols}")

tab_train_encoded = pd.get_dummies(tab_train, columns=cat_cols, dtype=np.float32)
tab_val_encoded = pd.get_dummies(tab_val, columns=cat_cols, dtype=np.float32)
tab_test_encoded = pd.get_dummies(tab_test, columns=cat_cols, dtype=np.float32)

tab_val_encoded = tab_val_encoded.reindex(columns=tab_train_encoded.columns, fill_value=0)
tab_test_encoded = tab_test_encoded.reindex(columns=tab_train_encoded.columns, fill_value=0)



Categorical columns: ['side_eye', 'sex']
Numerical columns: ['BScan', 'age', 'num_current_visit']


In [4]:

train_transform = transforms.Compose([
    transforms.Resize((420, 420)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((420, 420)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [5]:

class CustomImageDataset(Dataset):
    def __init__(self, dataframe, root_dir, tab_data, transform=None):
        self.dataframe = dataframe
        self.root_dir = root_dir
        self.tab_data = torch.tensor(tab_data.values, dtype=torch.float32)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['image'])
        localiser_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['LOCALIZER'])

        image = Image.open(img_name).convert('RGB')
        localiser = Image.open(localiser_name).convert('RGB')

        if self.transform:
            image = self.transform(image)
            localiser = self.transform(localiser)

        label = self.dataframe.iloc[idx]['label']
        tab = self.tab_data[idx]
        return image, localiser, tab, label

In [6]:
train_ds = CustomImageDataset(df_train, os.path.join(path, 'train'), tab_train_encoded, transform=train_transform)  
val_ds = CustomImageDataset(df_val, os.path.join(path, 'val'), tab_val_encoded, transform=test_transform)  
test_ds = CustomImageDataset(df_test, os.path.join(path, 'test'), tab_test_encoded, transform=test_transform)  
  
labels = df_train['label'].values.astype(int)  
class_counts = np.bincount(labels)  
sample_weights = 1.0 / class_counts[labels]  
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))  
  
train_loader = DataLoader(train_ds, batch_size=12, sampler=sampler)  
val_loader = DataLoader(val_ds, batch_size=12, shuffle=False)  
test_loader = DataLoader(test_ds, batch_size=12, shuffle=False)

In [7]:
from torchvision import models


class ImageEncoder(nn.Module):
    def __init__(self, embed_dim=768):
        super().__init__()
        weights = models.EfficientNet_V2_S_Weights.DEFAULT
        self.backbone = models.efficientnet_v2_s(weights=weights)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        self.fc = nn.Linear(in_features, embed_dim)
        for p in self.backbone.parameters():
            p.requires_grad = True

    def forward(self, x):
        features = self.backbone(x)
        return self.fc(features)


class LocaliserEncoder(nn.Module):
    def __init__(self, embed_dim=768):
        super().__init__()
        weights = models.EfficientNet_V2_S_Weights.DEFAULT
        self.backbone = models.efficientnet_v2_s(weights=weights)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        self.fc = nn.Linear(in_features, embed_dim)
        for p in self.backbone.parameters():
            p.requires_grad = True

    def forward(self, x):
        features = self.backbone(x)
        return self.fc(features)


class TabularMLP(nn.Module):
    def __init__(self, tab_dim, d_model, dropout=0.3):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(tab_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, d_model),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, tab):
        return self.mlp(tab)  # (B, d_model)


class OCT_Pred(nn.Module):
    def __init__(self, n_labels, tab_dim, embed_dim=768, d_model=None, dropout=0.3):
        super().__init__()
        if d_model is None:
            d_model = embed_dim

        self.image_encoder = ImageEncoder(embed_dim)
        self.localiser_encoder = LocaliserEncoder(embed_dim)
        self.tab_encoder = TabularMLP(tab_dim, d_model, dropout)

        fusion_dim = 2 * embed_dim + d_model

        self.mlp = nn.Sequential(
            nn.Linear(fusion_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, n_labels),
        )

    def forward(self, image, localiser, tab):
        img_features = self.image_encoder(image)
        loc_features = self.localiser_encoder(localiser)
        tab_features = self.tab_encoder(tab)

        combined = torch.cat([img_features, loc_features, tab_features], dim=-1)
        output = self.mlp(combined)
        return output, combined


class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(device)

In [8]:
from sklearn.metrics import classification_report
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_predicted = []

    pbar = tqdm(loader, desc="Training", leave=False)
    for images, localisers, tab, labels in pbar:
        images = images.to(device)
        localisers = localisers.to(device)
        tab = tab.to(device)
        labels = labels.long().to(device)

        optimizer.zero_grad()
        outputs, _ = model(images, localisers, tab)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

        all_labels.extend(labels.cpu().numpy())
        all_predicted.extend(predicted.cpu().numpy())

    accuracy = 100. * correct / total
    print(classification_report(all_labels, all_predicted))
    return running_loss / len(loader), accuracy

In [ ]:
from sklearn.metrics import f1_score, matthews_corrcoef, confusion_matrix, cohen_kappa_score, classification_report
import numpy as np

def specificity(class_ground_truth, class_prediction):
    eps = 0.000001
    cnf_matrix = confusion_matrix(class_ground_truth, class_prediction)
    FP = cnf_matrix.sum(axis=0) - np.diag(cnf_matrix)
    FN = cnf_matrix.sum(axis=1) - np.diag(cnf_matrix)
    TP = np.diag(cnf_matrix)
    TN = cnf_matrix.sum() - (FP + FN + TP)
    FP, FN, TP, TN = FP.astype(float), FN.astype(float), TP.astype(float), TN.astype(float)
    spe = TN / (TN + FP + eps)
    return spe.mean()

def evaluate(model, dataloader, criterion, device, show_report=True):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Evaluation', leave=False)
        for images, localisers, tab, labels in pbar:
            images = images.to(device)
            localisers = localisers.to(device)
            tab = tab.to(device)
            labels = labels.long().to(device)

            outputs, _ = model(images, localisers, tab)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = 100. * correct / total
    f1 = f1_score(all_labels, all_preds, average='micro')
    rk_corr = matthews_corrcoef(all_labels, all_preds)
    spec = specificity(all_labels, all_preds)
    qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    score = 0.1 * f1 + 0.1 * spec + 0.6 * qwk + 0.2 * rk_corr

    if show_report:
        print("\n" + "="*80)
        print("CLASSIFICATION REPORT - Per-Class Performance:")
        print("="*80)
        print(classification_report(all_labels, all_preds,
                                    target_names=['Class 0', 'Class 1', 'Class 2'],
                                    digits=4))
        print("="*80 + "\n")

    return {
        'loss': total_loss / len(dataloader),
        'accuracy': accuracy,
        'F1_score': f1,
        'Rk-correlation': rk_corr,
        'Specificity': spec,
        'Quadratic-weighted_Kappa': qwk,
        'score': score
    }




In [ ]:
#class FocalLoss(nn.Module):
#import torch.nn.functional as F
#    def __init__(self, gamma=2):
#        super().__init__()
#        self.gamma = gamma
#        
#    def forward(self, inputs, targets):
#        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
#        pt = torch.exp(-ce_loss)
#        focal_loss = (1 - pt) ** self.gamma * ce_loss
#        return focal_loss.mean()
#criterion = FocalLoss(gamma=2)


In [11]:
model = OCT_Pred(n_labels=n_labels, tab_dim=7).to(device)
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)
num_epochs = 5

best_score = -float('inf')

for epoch in range(num_epochs):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"{'='*50}")

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_metrics['loss']:.4f} | Val Acc: {val_metrics['accuracy']:.2f}%")
    print(f"Val Score: {val_metrics['score']:.4f}")

    if val_metrics['score'] > best_score:
        best_score = val_metrics['score']
        best_model=model
        print(f"New best model saved! Score: {best_score:.4f}")


Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 239MB/s]



Epoch 1/5


              precision    recall  f1-score   support

           0       0.69      0.93      0.79      2782
           1       0.92      0.37      0.53      2668
           2       0.77      0.97      0.86      2632

    accuracy                           0.75      8082
   macro avg       0.80      0.75      0.73      8082
weighted avg       0.79      0.75      0.73      8082




CLASSIFICATION REPORT - Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.0688    0.1083    0.0842       351
     Class 1     0.7472    0.8192    0.7815      2821
     Class 2     0.0000    0.0000    0.0000       650

    accuracy                         0.6146      3822
   macro avg     0.2720    0.3092    0.2886      3822
weighted avg     0.5578    0.6146    0.5846      3822


Train Loss: 0.2245 | Train Acc: 75.49%
Val Loss: 1.3213 | Val Acc: 61.46%
Val Score: 0.0503
New best model saved! Score: 0.0503

Epoch 2/5


...


CLASSIFICATION REPORT - Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.0504    0.0541    0.0522       351
     Class 1     0.7225    0.8564    0.7838      2821
     Class 2     0.1881    0.0292    0.0506       650

    accuracy                         0.6421      3822
   macro avg     0.3203    0.3133    0.2955      3822
weighted avg     0.5699    0.6421    0.5919      3822


Train Loss: 0.0377 | Train Acc: 95.83%
Val Loss: 1.5936 | Val Acc: 64.21%
Val Score: 0.1371
New best model saved! Score: 0.1371

Epoch 4/5


              precision    recall  f1-score   support

           0       0.97      1.00      0.98      2696
           1       1.00      0.96      0.98      2657
           2       0.99      1.00      1.00      2729

    accuracy                           0.98      8082
   macro avg       0.98      0.98      0.98      8082
weighted avg       0.98      0.98      0.98      8082




CLASSIFICATION REPORT - Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.0000    0.0000    0.0000       351
     Class 1     0.7308    0.9631    0.8310      2821
     Class 2     0.0000    0.0000    0.0000       650

    accuracy                         0.7109      3822
   macro avg     0.2436    0.3210    0.2770      3822
weighted avg     0.5394    0.7109    0.6134      3822


Train Loss: 0.0160 | Train Acc: 98.43%
Val Loss: 1.9739 | Val Acc: 71.09%
Val Score: 0.1271

Epoch 5/5


              precision    recall  f1-score   support

           0       0.98      1.00      0.99      2701
           1       1.00      0.97      0.98      2636
           2       0.99      1.00      1.00      2745

    accuracy                           0.99      8082
   macro avg       0.99      0.99      0.99      8082
weighted avg       0.99      0.99      0.99      8082




CLASSIFICATION REPORT - Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.0000    0.0000    0.0000       351
     Class 1     0.7324    0.9709    0.8349      2821
     Class 2     0.0000    0.0000    0.0000       650

    accuracy                         0.7166      3822
   macro avg     0.2441    0.3236    0.2783      3822
weighted avg     0.5405    0.7166    0.6163      3822


Train Loss: 0.0142 | Train Acc: 98.96%
Val Loss: 1.8721 | Val Acc: 71.66%
Val Score: 0.1242


In [12]:
results = evaluate(best_model, test_loader, criterion, device)
print(f"Accuracy: {results['accuracy']:.2f}%")
print(f"F1 Score: {results['F1_score']:.4f}")
print(f"Quadratic Weighted Kappa: {results['Quadratic-weighted_Kappa']:.4f}")
print(f"Rk Correlation (Spearman): {results['Rk-correlation']:.4f}")
print(f"Specificity: {results['Specificity']:.4f}")
print(f"Score: {results['score']:.4f}")


CLASSIFICATION REPORT - Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     1.0000    0.0329    0.0637       578
     Class 1     0.8448    0.9960    0.9142      4810
     Class 2     0.0000    0.0000    0.0000       321

    accuracy                         0.8425      5709
   macro avg     0.6149    0.3430    0.3260      5709
weighted avg     0.8130    0.8425    0.7767      5709


Accuracy: 84.25%
F1 Score: 0.8425
Quadratic Weighted Kappa: 0.0406
Rk Correlation (Spearman): 0.0840
Specificity: 0.6725
Score: 0.1926


In [13]:
from google.colab import drive
drive.mount('/content/drive')
#from google.colab import files
#files.download('model.pth')
import torch
torch.save(best_model.state_dict(), '/content/drive/MyDrive/best_model_mlp.pth')

Mounted at /content/drive
